## 📎 Code this notebook uses
*(how to run + which code each step uses → [`README.md`](README.md);  file-by-file code reference → [`CODE_MANUAL.pdf`](CODE_MANUAL.pdf))*

- **Compute — `pipeline/compute/`:** `retrieve_kd`

> the K_d sweep + bootstrap (the headline result); run via `subprocess`


# 03 — Per-site K_d retrieval and bootstrap

This is the **main computation** of the paper. It runs the per-site
K_d sweep at Apollo 15 and 17 and the non-parametric bootstrap.

## Two ways to use this notebook

**FAST path**: Run only **Step 1** below. This regenerates the
canonical `results/kd_retrieval_results.json` with K_d*, bootstrap, Q_b
sensitivity, and joint (K_d, H) fit. That is enough to feed all
headline figures in notebooks 03 + 04.

**FULL path**: Also run **Step 2** to regenerate the
auxiliary sensitivity-sweep JSONs that feed Table 3 (per-component
error budget). These are independent computations and can be skipped
— their canonical outputs ship with the repository under `results/`.

Each script writes its result to a JSON file in `results/` and is
idempotent (re-running overwrites).

In [1]:
import sys, pathlib, subprocess, json, time
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

## Step 1 — the central retrieval

This runs `pipeline/compute/retrieve_kd.py`, the heart of the paper. In
plain terms, for **each site** it asks: *what value of the deep
conductivity K_d makes the modelled deep-sensor temperatures best match
the data?*

It does this by:
1. Loading the bundled HFE record (the stable-window temperatures)
2. **Sweeping** K_d across a grid of trial values, and for each value
   solving the 1-D heat model to its settled state (via `run_with()`)
3. Finding the K_d that minimises the deep-sensor RMSE — that minimum is
   **K_d\***
4. Repeating on 1500 bootstrap resamples (with ±2.5 cm sensor-depth
   jitter) to get a confidence interval
5. Mapping the K_d/Q_b trade-off and the joint (K_d, H) fit
6. Writing everything to `results/kd_retrieval_results.json` and drawing
   the bootstrap + robustness figures

The cell below runs it as a subprocess and **streams its live tqdm progress
bars** so you can watch the sweep, bootstrap, and joint-grid stages advance
(instead of waiting ~90 min for one dump at the end).

In [ ]:
import io

# Stream the run live so you can watch the tqdm progress bars advance
# (this is the ~90 min headline computation). Notes:
#   * tqdm writes its bars to stderr, so we merge stderr into stdout.
#   * tqdm updates a bar IN PLACE with '\r'. We therefore read the pipe as
#     bytes and wrap it with newline='' — that keeps the '\r' intact (so each
#     bar overwrites its own line instead of spamming thousands of lines) and
#     still decodes the multibyte block characters correctly.
proc = subprocess.Popen(
    [sys.executable, str(ROOT / 'pipeline' / 'compute' / 'retrieve_kd.py')],
    cwd=str(ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
)
stream = io.TextIOWrapper(proc.stdout, encoding='utf-8', errors='replace', newline='')
for ch in iter(lambda: stream.read(1), ''):
    sys.stdout.write(ch)
    sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'retrieve_kd failed (exit {proc.returncode})')
print('\nRetrieval complete.')

### Verify the headline result

Reads the JSON just written and prints, per site: the retrieved **K_d\***,
its RMSE, and the bootstrap **median and 95% confidence interval**. The
cell also prints the expected values from the paper so you can confirm
your run reproduced them.

**What to look for:** A17's K_d\* (~7.1) sits above A15's (~4.6) — a
contrast of roughly 1.5×. The two confidence intervals overlap, and about
97% of the bootstrap resamples favour A17, so the contrast is real but
marginal (not significant at the 95% level).

In [3]:
d = json.loads((ROOT / 'results' / 'kd_retrieval_results.json').read_text())
for site in ('A15', 'A17'):
    s = d[site]; b = s['bootstrap']
    print(f'  {site}:')
    print(f'    K_d*        = {s["kd_star"]*1e3:.3f} mW m^-1 K^-1')
    print(f'    RMSE at K_d*= {s["rmse_star"]:.3f} K')
    print(f'    bootstrap median = {b["median"]*1e3:.2f} mW m^-1 K^-1')
    print(f'    95% CI      = [{b["ci_lo"]*1e3:.2f}, {b["ci_hi"]*1e3:.2f}] mW m^-1 K^-1')
    print()
print('Expected (matches paper):')
print('  A15  K_d* = 4.60 mW m^-1 K^-1, CI [4.18, 6.96]')
print('  A17  K_d* = 7.08 mW m^-1 K^-1, CI [6.16, 8.07]')

  A15:
    K_d*        = 4.600 mW m^-1 K^-1
    RMSE at K_d*= 1.000 K
    bootstrap median = 4.63 mW m^-1 K^-1
    95% CI      = [4.18, 6.96] mW m^-1 K^-1

  A17:
    K_d*        = 7.079 mW m^-1 K^-1
    RMSE at K_d*= 0.398 K
    bootstrap median = 7.08 mW m^-1 K^-1
    95% CI      = [6.16, 8.07] mW m^-1 K^-1

Expected (matches paper):
  A15  K_d* = 4.60 mW m^-1 K^-1, CI [4.18, 6.96]
  A17  K_d* = 7.08 mW m^-1 K^-1, CI [6.16, 8.07]


## Step 2 — OPTIONAL: auxiliary sensitivity sweeps

These regenerate the JSONs that feed Table 3 (per-component error
budget). Each one runs additional K_d sweeps under perturbed inputs.

**Skip this whole step if you only want the headline figures.** The
canonical outputs are already in `results/`.

Each script re-runs the retrieval under one perturbed assumption and
writes its own JSON:

| Script | What it does → output |
|---|---|
| `compute_borestem_sensitivity.py` | re-retrieves K_d while sweeping the borestem exclusion depth → `borestem_sensitivity.json` |
| `compute_stability_threshold_sensitivity.py` | varies the stable-window slope threshold and reports how K_d\* moves → `stability_threshold_sensitivity.json` |
| `compute_surface_bias_test.py` | perturbs the Bond albedo to test whether a surface-temperature mismatch biases the deep K_d → `surface_bias_test.json` |
| `compute_uniform_kd_sensitivity.py` | fine per-site-K_d vs shared-K_d/free-Q_b model comparison → `uniform_kd_test.json` (Table 4 inputs) |
| `compute_headline_rmse.py` | deep-sensor RMSE under global Hayne, per-site fit, and Martínez forward → `headline_rmse.json` (Table 2) |
| `compute_model_selection.py` | reduced-χ² and AICc per conductivity model → `model_selection.json` |
| `compute_error_budget.py` | combines the retrieval and the sensitivity JSONs in quadrature → `kd_error_budget.json` (Table 3) |

Set `RUN_AUXILIARY = True` to execute. Default is False so the notebook
runs end-to-end without the extra sweeps.

In [4]:
RUN_AUXILIARY = False    # change to True to regenerate all auxiliary JSONs

if RUN_AUXILIARY:
    scripts = [
        'compute_borestem_sensitivity.py',
        'compute_stability_threshold_sensitivity.py',
        'compute_surface_bias_test.py',
        'compute_uniform_kd_sensitivity.py',
        'compute_headline_rmse.py',
        'compute_model_selection.py',
        'compute_error_budget.py',
    ]
    for script in scripts:
        print(f'\n=== {script} ===')
        r = subprocess.run(
            [sys.executable, str(ROOT / 'pipeline' / 'compute' / script)],
            cwd=str(ROOT), capture_output=True, text=True,
        )
        if r.returncode != 0:
            print('  FAILED:', r.stderr[-300:])
        else:
            print('  ok')
else:
    print('Auxiliary sweeps SKIPPED (RUN_AUXILIARY = False).')
    print('Canonical JSONs are already in results/. Set RUN_AUXILIARY = True')
    print('above and re-run this cell to regenerate them.')

Auxiliary sweeps SKIPPED (RUN_AUXILIARY = False).
Canonical JSONs are already in results/. Set RUN_AUXILIARY = True
above and re-run this cell to regenerate them (~50 min total).


## Step 3 — OPTIONAL: Bayesian MCMC cross-check

`bayesian_crosscheck.py` runs an `emcee` MCMC over a direct 2-D
RMSE(K_d, Q_b) surface at each site, giving the joint (K_d, Q_b)
posterior. It is an independent cross-check of the *direction* of the
inter-site contrast (not its magnitude), and feeds no headline figure.
Skip it if you don't need the Bayesian discussion section.

In [5]:
RUN_MCMC = False    # change to True to run the MCMC cross-check

if RUN_MCMC:
    r = subprocess.run(
        [sys.executable, str(ROOT / 'pipeline' / 'compute' / 'bayesian_crosscheck.py')],
        cwd=str(ROOT), capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('Phase B MCMC failed:', r.stderr[-500:])
    else:
        print('Phase B MCMC complete.')
else:
    print('MCMC cross-check SKIPPED (RUN_MCMC = False).')

MCMC cross-check SKIPPED (RUN_MCMC = False).


---

**Next**: open `04_results.ipynb` to render Figs 5-9 and Tables 2-3.